# 산불감지 OOD 재설계 학습 — Colab L4

**데이터**: 영상 30편 다양화 + 스타일 균형 (~1,500장)
**설정**: E (대칭 강증강), 7모델 x 3fold, epoch 15
**예상**: L4 기준 ~2시간 (T4 ~4시간). 끊겨도 Cell6 재실행 시 이어학습.

실행: Cell 1→2→3→5→6  (Cell 4 Kaggle인증 불필요)

In [ ]:
# Cell 1: GPU + 코드 클론
import torch
assert torch.cuda.is_available(), 'GPU 런타임 설정 필요'
print('GPU:', torch.cuda.get_device_name(0))
!git clone https://github.com/yuntaewon812/fireimage_detection.git /content/fireimage_detection
%cd /content/fireimage_detection

In [ ]:
# Cell 2: Drive 연결 (가중치·결과 영구저장)
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/fireimage_ablation'
for sub in ['model_save', 'results']:
    os.makedirs(f'{CKPT}/{sub}', exist_ok=True)
    link = f'/content/fireimage_detection/{sub}'
    if os.path.islink(link): os.unlink(link)
    elif os.path.exists(link): shutil.rmtree(link, ignore_errors=True)
    os.symlink(f'{CKPT}/{sub}', link)
import glob
done = glob.glob(f'{CKPT}/model_save/**/*.pt', recursive=True)
print(f'기존 완료 가중치: {len(done)}개')

In [ ]:
# Cell 3: 패키지
!pip install timm einops transformers kaggle -q
print('완료')

In [ ]:
# Cell 4: Kaggle 인증
import os, getpass, re
raw = getpass.getpass('Kaggle API Token (KGAT_...): ')
token = re.sub(r'[^A-Za-z0-9_\-]', '', raw)
print('토큰 길이:', len(token))
os.environ['KAGGLE_API_TOKEN'] = token
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
open(os.path.expanduser('~/.kaggle/access_token'), 'w').write(token)
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
!kaggle datasets list --user yuntarwon

In [ ]:
# Cell 5: 재설계 데이터 로드 (Drive zip 압축해제)
# ※ Kaggle 다운로드 대신 fireimage_clean.zip 사용
import zipfile, os, glob
BASE = "/content/fireimage_detection/data/fireimage"
os.makedirs(BASE, exist_ok=True)
ZIP = "/content/drive/MyDrive/fireimage_clean.zip"
assert os.path.exists(ZIP), f"Drive에 zip 없음: {ZIP} — MyDrive에 업로드하세요"
with zipfile.ZipFile(ZIP) as z:
    z.extractall(BASE)
IMG = (".jpg",".jpeg",".png",".bmp")
n = sum(1 for f in glob.glob(f"{BASE}/normal/**/*", recursive=True) if f.lower().endswith(IMG))
a = sum(1 for f in glob.glob(f"{BASE}/abnormal/**/*", recursive=True) if f.lower().endswith(IMG))
print(f"압축해제 완료 — normal {n} / abnormal {a}")
assert n > 0 and a > 0

In [ ]:
# Cell 6: 재설계 데이터로 단일 학습 (설정 E, epoch 15)
%cd /content/fireimage_detection
!git pull origin main   # 최신 코드 (epoch CLI, 대칭증강)

# 7모델 x 3fold x 1설정 = 21회, ~1,600장, epoch15 → 약 3~5시간
!python main_ablation.py --class_name fireimage --setting E --epochs 15 --patience 5

In [ ]:
# Cell 7: OOD F1 비교표 (A~E)
import pandas as pd, os
CSVS = {
    "A(baseline)": "results/fireimage/metrics.csv",
    "B(pretrained)": "results/fireimage_abl_B/metrics.csv",
    "C(+augment)": "results/fireimage_abl_C/metrics.csv",
    "D(+mixup)": "results/fireimage_abl_D/metrics.csv",
    "E(+YT augment)": "results/fireimage_abl_E/metrics.csv",
}
dfs = {k: pd.read_csv(v) for k, v in CSVS.items() if os.path.exists(v)}
models = ["Resnet50","DenseNet121","efficientnetv2","efficientnetv2_proposal",
          "nextvit","maxvit","internimage"]
rows = []
for m in models:
    row = {"model": m}
    for k, df in dfs.items():
        vals = []
        for i in range(3):
            r = df[df["model name"] == f"{m}_{i}"]
            if not r.empty:
                vals.append(float(r["F1 score"].iloc[0].split("(")[0]))
        row[f"{k}_OOD"] = f"{min(vals):.3f}" if vals else "-"
    rows.append(row)
comp = pd.DataFrame(rows).set_index("model")
print("=== OOD fold F1 비교 (YouTube test fold, 낮을수록 취약) ===")
print(comp.to_string())

In [ ]:
# Cell 8: 결과 다운로드 (가중치는 Drive에)
import shutil
shutil.make_archive('/content/ablation_results', 'zip',
                    '/content/fireimage_detection', 'results')
from google.colab import files
files.download('/content/ablation_results.zip')